In [ ]:
# NOTA IMPORTANT: PERQUÈ FUNCIONI CAL OBRIR JUPYTER DES D'ANACONDA AMB L'ORDRE:

# jupyter notebook --NotebookApp.iopub_data_rate_limit=1.0e8

# Si no, no calcula bé el vocabulari.

# IMPORTACIÓN DE LIBRERÍAS

import os 
import glob
import re
import nltk
import emoji

from bs4 import BeautifulSoup
from bs4.element import Comment
from unicodedata import name   # Ver https://www.compart.com/en/unicode/U+1F321
from math import log10
from pathlib import Path

In [ ]:
# PARÁMETROS

# Ruta a los ficheros html
folder = "input\\files"
folder_output = "output"
path_to_files = folder + "\\*"

# Etiquetas con contenido semántico relevante (tras un análisis manual de los primeros 3 textos)
relevant_tags = [
    'article',
    'meta (name="description")',
    'meta (name="keywords")',
    'meta (property="article:tag")',
    'meta (name="news_keywords")'
]

In [ ]:
# IMPORTACIÓN DE LOS TEXTOS DEL CORPUS

def get_documents(path_to_files):
    
    """
    Crea una lista de textos completos en formato original a partir de los ficheros de una carpeta
    
    Parámetros
    - path_to_files: Ruta a los ficheros. En caso de querer incluir todos ficheros de una carpeta,
      la ruta debe terminar en /* 
    """
    
    files = glob.glob(path_to_files)
    htmls = list(range(len(files)))
    titles = []

    for i, file in enumerate(files):
        title = file.replace(folder + "\\", '')
        titles.append(title)
        with open(file, 'r', encoding='utf-8') as html:
            htmls[i] = html.read()
    
    return titles, htmls


In [ ]:
# CONVERSIÓN HTML A TEXTO PLANO

def tag_visible(element):
    """
    Devuelve false or False según si el elemento html que se pasa como parámetro está o no en
    la lista negra
    """
    
    tag_blacklist = [
        '[document]',
        'noscript',
        'header',
        'html',
        'meta',
        'head', 
        'input',
        'script',
        'style'
    ]    

    if element.parent.name in tag_blacklist:
        return False
    
    if isinstance(element, Comment):
        return False
      
    return True
'''
def text_from_html(html):
    """
     rehace el texto con los textos seleccionados, eliminando saltos de línea.
    """

    soup = BeautifulSoup(html, 'html.parser')  # Se crea un objeto BeautifulSoup.
    text = soup.findAll(text=True) # Se seleccionan las cadenas de texto, sin las etiquetas.
    visible_text = filter(tag_visible, text)  # Se seleccionan las cadenas de texto cuya etiqueta padre es visible.
    return u" ".join(t.strip() for t in visible_text)  # Se concatenan los textos, sin los saltos de línea.
'''

def text_from_html(html):
    """
     rehace el texto con los textos seleccionados, eliminando saltos de línea.
    """
    text_title = ''   # Inicializamos variables para que existan, aunque vacías
    text_meta = ''
    text_article = ''
    
    soup = BeautifulSoup(html, 'html.parser')  # Se crea un objeto BeautifulSoup.
    
    try:
        article = soup.article.findAll(text=True) # Se seleccionan las cadenas de texto, sin las etiquetas.    
        visible_article = filter(tag_visible, article)  # Se seleccionan las cadenas de texto cuya etiqueta padre es visible.
        text_article = u" ".join(t.strip() for t in visible_article)  
    except:
        pass

    try:
        title = soup.title.findAll(text=True) # Se seleccionan las cadenas de texto, sin las etiquetas.    
        visible_title = filter(tag_visible, title)  # Se seleccionan las cadenas de texto cuya etiqueta padre es visible.
        text_title = u" ".join(t.strip() for t in visible_title)      
    except:
        pass
    
    lists_meta = []
    lists_meta.append(soup.findAll('meta', attrs={'name':'description'}))
    lists_meta.append(soup.findAll('meta', attrs={'name':'keywords'}))
    lists_meta.append(soup.findAll('meta', attrs={'name':'description'}))
    lists_meta.append(soup.findAll('meta', attrs={'property':'article:tag'}))

    text_meta = ''
    for list_ in lists_meta:
        for tag in list_:
            text_meta += tag['content'] + ' '
    text = text_meta + ' ' + text_title + ' ' + text_article
    
    return text


In [ ]:
# LIMPIEZA

def clean(text):
    '''
    Preprocesamiento del texto, anterior a la tokenización, para:
    - Eliminar cifras
    - Detectar emojis
    '''
    
    pattern = r'''(?x)
       | \W-?\d+(?:[:.,]\d+)*(?:-?\w)*       # cifras enteras o decimales que no forman parte de palabras tipo "b2b"
       | (?:\d{1,2}[\/\.\-]\d{1,2}[\/\.\-]\d{4})         # fechas en formato 04/11/2020 o 4/11/2020
       | (?:\d{1,2}\/\d{1,2}\/\d{4})         # fechas en formato 04/11/2020 o 4/11/2020
         '''
    
    text = re.sub(pattern, '', text)
    
    text = emoji.demojize(text)
    
    return text


In [ ]:
print(clean('fecha: co12'))

In [ ]:
# TOKENIZACION

def tokenize(text):
    '''
    Convierte un texto en una lista de tokens, siguiendo las siguientes reglas de normalización:
    
    1)  Patrones aceptados 
    
        Se incluyen como expresiones regulares entre paréntesis se parados por 1 (OR)
        La expresión entre paréntesis empieza por (?: para evitar gasto innecesario de memoria
        (esta cadena previa indica que la expresión que le sigui no debe guardada en memoria como una variable "backward")
        
        Los patrones aceptados son cadenas de:
        - Palabras y palabras compuestas: caracteres alfabéticos y guiones bajos, con guiones medios intercalados opcionales:
          \w+(-\w+)*
        - Abreviaciones con puntos intercalados, tipo U.S.A.
          \([A-Z]\.)+
        
        Podría ser deseable incluir, pero no se ha hecho:
        - Símbolos y pictogramas unicode con significado (el símbolo o el nombre del símbolo)
            - rango 1F300—1F5FF
              
        (si el idioma fuese inglés, podría ser importante separar contracciones)
    
    
    
    Parámetros
    - text: Texto a tokenizar, SIN SALTOS DE LÍNIA.
    '''
    
    # tokens que aceptaremos (de nltk.org/book/ch03). Output: cadena de palabras.
   
        
    pattern = r'''(?ux)        # permitir "verbose"
        (?:[A-Z]\.)+           # abreviaciones tipo U.S.A. (?: needs to be added to specify that the parenthesis specify the scope of the disjunction, not the selection o strings to be extracted)
       | [\w\d]+(?:-[\w\d]+)*          # palabras que aceptan guiones interiores opcionales
       |(?:[DSds][Rr][Aa]?\.)            # Abreviacoes do tipo Sr. Sra., Dr. Dra. (NO FUNCIONA)        
       | (?::\w+)(?:_\w+)*              # palabras precedidas de : (emojis traducidos por el módulo Emojis)
    '''
    
    '''
    AMPLIAR.
    - Analizar errores
    - Alguna solución para detectar phrasal verbs?
    '''

    # tokenización. Output: lista
   
    tokens = nltk.regexp_tokenize(text, pattern, gaps=False)
    
    return tokens
    
    

In [ ]:
tokenize('3')

In [ ]:
# NORMALIZACIÓN

def to_lowercase(tokens):
    '''
    Convierte todos los tokens de una lista a minúscula.
    '''
    
    tokens = [token.lower() for token in tokens]   
  
    return tokens

def remove_dots(tokens):
    '''
    Elimina los puntos de las abreviaciones
    '''

    tokens = [re.sub('\.', '', token) for token in tokens]   
  
    return tokens

def remove_stopwords(tokens):
    """
    Elimina las stopwords del español incluidas en NLTK de una lista de tokens
    
    Parámetros:
    - tokens: lista de tokens
    """
    
    stopwords = nltk.corpus.stopwords.words('spanish')
    stopwords.sort()
    tokens = [token for token in tokens if token not in stopwords]

    return(tokens) 

In [ ]:
remove_dots(['U.S.A', 'hola'])

In [ ]:
# TRUNCADO

def stem(tokens):
    '''
    Parámetros
    - tokens: lista de palabras o caracteres a truncar
    '''
    stemmer = nltk.stem.SnowballStemmer('spanish')
    stemmed_text = [stemmer.stem(token) for token in tokens]
    return stemmed_text


In [ ]:
# VOCABULARIO Y FRECUENCIAS

# VOCABULARIO

def vocabulary(tokens):
    '''
    Devuelve una lista de tokens únicos ordenados alfabéticamente a partir de una lista de tokens
    '''
    vocabulary = list(set(tokens))
    vocabulary.sort()
    return vocabulary

# FRECUENCIAS DE LOS TOKENS

def toks_frequencies(tokens):
    '''
    Devuelve un diccionario con los tokens únicos de la lista de entrada y la frecuencia con que 
    se da cada uno de ellos
    
    Parámetros:
    - Lista de tokens
    '''
    toks_frequencies_tuples = nltk.FreqDist(tokens).most_common(n=None)
    toks_frequencies_tuples.sort(key=lambda tup:tup[0])    
    
    toks_frequencies = {}
    for token, freq in toks_frequencies_tuples:
        toks_frequencies[token] = freq

    return toks_frequencies           

# FRECUENCIAS DE LOS DOCUMENTOS

def docs_frequencies(vocabulary, docs_tokens):
    '''
    Diccionarios con los tokens del vocabulario y el número de documentos del corpus
    en que aparece el token correspondiente.
    
    Parámetros:
    - vocabulary: lista de tokens con el vocabulario completo del corpus, ordenado alfabéticamente.
    - docs_tokens: lista de listas de tokens, cada una con los tokens de cada documento.
    '''
    docs_frequencies = {}
    
    for token in vocabulary:
        docs_frequencies[token] = 0
        for j in range(len(docs_tokens)):
            if token in docs_tokens[j]:
                docs_frequencies[token] += 1
    return docs_frequencies


def corpus_matrix(vocabulary, docs_tokens):
    '''
    Devuelve una lista de diccionarios. Cada diccionario representa un documento y contiene como claves todas 
    las palabras del diccionario, con valor igual al número de veces en que se da la palabra en el documento. 
    
    Parámetros
    - vocabulary: lista con el vocabulario 
    - docs_tokens: lista de listas, donde cada lista son los tokens de un documento.
    
    Salida
    - Lista de diccionarios token:frecuencia_en_documento
    '''
     
    corpus_matrix = []

    for doc_tokens in docs_tokens:
        doc_dict = {}
        tok_freqs = toks_frequencies(doc_tokens)

        for i, token in enumerate(vocabulary):
            if token in tok_freqs.keys():
                doc_dict[token] = tok_freqs[token]

        corpus_matrix.append(doc_dict)
        
    return corpus_matrix

# FICHERO INVERTIDO

def inverted(vocabulary, corpus_matrix):
    '''
    Crea una lista de listas, una para cada palabra del vocabulario y todas de dimensión igual
    al número de documentos del corpus. Los elementos de las listas son las frecuencias del token que representan
    para cada documento
    '''

    inverted = ''

    for token in vocabulary:
        print("token:" + token + "************" )
        freqs_in_docs = [] 
        for i, doc in enumerate(corpus_matrix):
            print("    Documento:" + str(i) + "************" )
            print(corpus_matrix[i])
            print("************")
            if token in doc.keys():
                print("    yes")
                freqs_in_docs.append(str(doc[token]))
            else:
                print("    no")
                freqs_in_docs.append('0')
            print("    Lista de frecuencias:" + str(freqs_in_docs) + "************" )
        string = token + ' -> ' 
        for i, freq in enumerate(freqs_in_docs):
            string = string + 'doc_' + str(i) + ":" + freq + ' '
        inverted += string + '\n'
    return inverted



In [ ]:
# FUNCIONES DE PESADO

# LOCALES - BINARY

def binary(corpus_voc, doc_vocabulary):
    '''
    Crea el vector representación de un documento en el espacio vectorial definido por un vocabulario, con 
    pesos definidos según la función binaria
    
    Parámetros:
    - corpus_voc: lista de tokens correspondiente al vocabulario del corpus que define el espacio vectorial
    - doc_voc: lista de tuplas (token, frecuencia con que aparece el token) en el documento a representar.    
        
    Salida
    - doc_vec: vector d
    '''
    
    doc_vec = []
    
    for token in corpus_voc:
        if token in doc_vocabulary:
            doc_vec.append(1)
        else:
            doc_vec.append(0)
            
    return doc_vec

# LOCALES - TERM FREQUENCY

def tf(corpus_voc, doc_dict):
    '''
    Crea el vector representación de un documento en el espacio vectorial definido por un vocabulario, con 
    pesos definidos según la función 'term frequency'
    
    Parámetros:
    - corpus_matrix: Diccionario cuyas claves son los tokens del vocabulario que aparecen en el documento 
    y los valores el número de veces en que aparecen.
        
    Salida
    - doc_vec: vector representación
    '''
    
    doc_vec = []
    
    for token in corpus_voc:
        if token in doc_dict.keys():
            doc_vec.append(doc_dict[token])
        else:
            doc_vec.append(0)
        
    return doc_vec

# LOCAL - WEIGHTED TERM FREQUENCY

def wtf(corpus_voc, doc_dict):
    '''
    Crea el vector representación de un documento en el espacio vectorial definido por un vocabulario, con 
    pesos definidos según la función 'weigthed term frequency'
    
    Parámetros:
    - corpus_matrix: Diccionario cuyas claves son los tokens del vocabulario que aparecen en el documento 
    y los valores el número de veces en que aparecen.
        
    Salida
    - doc_vec: vector representación
    '''
    
    doc_vec = tf(corpus_voc, doc_dict)
    doc_vec = [coordinate / sum(doc_vec) for coordinate in doc_vec]
    
        
    return doc_vec
    
# GLOBALES

# GLOBAL - FRECUENCIA INVERSA DEL DOCUMENTO
            
def bin_idf(docs_frequencies, doc_dict):
    '''
    Parámetros:
    - docs_frequencies_: diccionario con los tokens del vocabulario y el número de documentos del corpus
      en que aparecen cada uno de ellos.
    - doc_dict: diccionario con los tokens que aparecen en el documento a representar como claves, y la frecuencia
    
    Salida
    - doc_vec: vector de dimensión igual a la longitud del vocabulario, con enteros indicando el peso de cada token.
    '''

    doc_vec = []

    for i, key in enumerate(docs_frequencies):   
        if key in doc_dict:
            f = 1 + log10(len(docs_frequencies) / docs_frequencies[key])
        else:
            f = 0

        doc_vec.append(f)
            
    return doc_vec    
        
# FRECUENCIA DEL TÉRMINO X FRECUENCIA INVERSA DEL DOCUMENTO (tf-idf)

            
def tf_idf(docs_frequencies, doc_dict):
    '''
    Parámetros:
    - docs_frequencies_: diccionario con los tokens del vocabulario y el número de documentos del corpus
      en que aparecen cada uno de ellos.
    - doc_dict: diccionario con los tokens que aparecen en el documento a representar como claves, y la frecuencia
    
    Salida
    - doc_vec: vector de dimensión igual a la longitud del vocabulario, con enteros indicando el peso de cada token.
    '''

    doc_vec = []

    for i, key in enumerate(docs_frequencies):   
        if key in doc_dict:
            f = doc_dict[key] * log10(len(docs_frequencies) / docs_frequencies[key])
        else:
            f = 0

        doc_vec.append(f)
            
    return doc_vec    

In [ ]:
# EXPORTACIÓN

def export_vector(subfolder, num, title, vector):
    '''
    Exporta el contenido de "vector" a ficheros txt localizados en la carpeta "subfolder". El nombre de cada 
    fichero es incluye:
    - La palabra "Doc" y el número de documento
    - El nombre de la carpeta
    - El nombre del fichero html original
    
    Parámetros
    - subfolder: ruta a la carpeta donde deben guardarse los ficheros
    - num: numero de documento
    - titles: Cadena de texto con el nombre  del fichero html original
    - vector: lista (puede ser el vector representación de un documento u otra cosa)
    
    Salida
    - ficheros con los elementos de la lista separados por espacio en blanco.
    
    ''' 
    
    folder = folder_output + '\\' + subfolder
    export_path = folder + "\\Doc" + str(num) + subfolder + title + ".txt"

    print(folder)
    if not os.path.exists(folder):
        try:
            os.mkdir(folder)
        except: 
            print("No se ha podido crear el la carpeta" + subfolder)
        else:
            print("Se ha creado la carpeta " + subfolder)

    with open(export_path, "w+", encoding="utf-8") as output:
        for item in vector:
            output.write(" " + str(item))

    print("****** Fichero de salida:")
    print(export_path)
    print("****** Vector documento (vector de la dimensión del vocabulario, con los pesos para el documento")    
    print(vector)            


In [ ]:
# EJECUCIÓN DE LA REPRESENTACIÓN VECTORIAL

titles, htmls = get_documents(path_to_files)
'''
print("****** Titles (ficheros del corpus):")
print(titles)
print("****** htmls (contenido de los ficheros del corpus):")
print(htmls)
'''
N = len(titles)
print("****** N (número de documentos del corpus):")
print(N)

texts = [None] * len(htmls)  # Lista de textos planos extraídos de los documentos html
docs_tokens = [None] * len(htmls)  # Lista de listas de tokens, cada una del tamaño del documento al que corresponda.
all_tokens = []   # Todos los tokens de todos los documentos en una única lista.

for i, html in enumerate(htmls):
    print("html" + str(i))
    print(html[0:100])
    print(titles[i])
    text = text_from_html(html)
    print("text" + str(i))
    print(text[0:100])
    texts[i] = text
    text = clean(text)
#    print("Cleaned text" + str(i))
#    print(text[0:100])
    tokens = tokenize(text)
    print("tokens" + str(i))
    print(tokens[0:20])
    tokens = to_lowercase(tokens)
#    print("tokens lowercase" + str(i))
#    print(tokens[0:20])
    tokens = remove_dots(tokens)
    tokens = remove_stopwords(tokens)
#    print("tokens without stopwords" + str(i))
#    print(tokens[0:20])
    stemmed_tokens = stem(tokens)
    print("stemmed_tokens" + str(i))
    print(stemmed_tokens[0:20])
    docs_tokens[i] = stemmed_tokens   
    print("docs_tokens" + str(i))
    print(docs_tokens[i][0:20])
    all_tokens += stemmed_tokens

# print('all_tokens')
# print(all_tokens)
    
vocabulary_ = vocabulary(all_tokens)
# vocabulary_ = all_tokens
toks_frequencies_ = toks_frequencies(all_tokens)
docs_frequencies_ = docs_frequencies(vocabulary_, docs_tokens)
corpus_matrix_ = corpus_matrix(vocabulary_, docs_tokens)

print("****** Documents tokens (lista de listas de tokens, cada una correspondiente a un documento tokenizado y normalizado)")
print(docs_tokens)
print("****** Vocabulary (lista de tokens única que incluye todos los tokens normalizados del corpus)")
print(vocabulary_)
print("****** Tokens frequencies (diccionario con el vocabulario y el número de veces en que aparece cada token en el corpus)")
print(toks_frequencies_)
print("****** Document frequencies (diccionario con el vocabulario y el número de documentos que incluyen cada token)")
print(docs_frequencies_)
print("****** Document matrix (lista de diccionarios, donde cada diccionario indica los tokens que contiene un documento y el número de veces en que aparecen")
print(corpus_matrix_)


In [ ]:
# EXPORTACIÓN DEL TEXTO EXTRAÍDO DEL HTML 

for i, text in enumerate(texts):
    subfolder = "texto_plano"
    folder = folder_output + '\\' + subfolder
#    export_path = folder + "\\Doc" + str(num) + subfolder + title + ".txt"    
    export_path = folder +  '\\Doc' + str(i) + "_" + subfolder + "_" + titles[i] + ".txt"
    print(export_path)
    
    if not os.path.exists(folder):
        try:
            os.mkdir(folder)
        except: 
            print("No se ha podido crear el la carpeta" + subfolder)
        else:
            print("Se ha creado la carpeta " + subfolder)

    with open(export_path, "w+", encoding="utf-8") as output:
        for line in text:
            output.write(str(line))

print("****** Fichero de salida:")
print(export_path)


In [ ]:
# EXPORTACIÓN VOCABULARIO 

export_vector("vocabulary", num='All ', title='', vector=vocabulary_)
print(vocabulary_)


In [ ]:
inverted_ = inverted(vocabulary_, corpus_matrix_)
print(inverted_)

In [ ]:
# EXPORTACIÓN FICHERO INVERTIDO

inverted_ = inverted(vocabulary_, corpus_matrix_)

subfolder = "fichero_invertido"
folder = folder_output + '\\' + subfolder
export_path = folder +  '\\' + subfolder + ".txt"

if not os.path.exists(folder):
    try:
        os.mkdir(folder)
    except: 
        print("No se ha podido crear el la carpeta" + subfolder)
    else:
        print("Se ha creado la carpeta " + subfolder)

with open(export_path, "w+") as output:
    for line in inverted_:
        output.write(str(line))

print("****** Fichero de salida:")
print(export_path)
print("****** Documento invertido")
print(inverted_)



In [ ]:
# BINARY REPRESENTATIONS (bin)

docs_vectors = [None] * len(htmls)  # Lista de listas de 0 y 1, todas del tamaño del vocabulario.

for i in range(len(docs_tokens)):
    docs_vectors[i] = binary(vocabulary_, docs_tokens[i])
    export_vector("bin", i, titles[i], docs_vectors[i])
    
print("****** Vectores documentos (lista de listas, o matriz, con todos los vectores documento)")
print(docs_vectors)

In [ ]:
# TERM FREQUENCY REPRESENTATIONS (tf)

docs_vectors = [None] * len(htmls)  # Lista de listas, todas del tamaño del vocabulario.

for i in range(len(corpus_matrix_)):
    docs_vectors[i] = tf(vocabulary_, corpus_matrix_[i])
    export_vector("tf", i, titles[i], docs_vectors[i])
    
print("****** Vectores documentos (lista de listas, o matriz, con todos los vectores documento)")
print(docs_vectors)

In [ ]:
# WEIGHTED TERM FREQUENCY REPRESENTATIONS (tf)

docs_vectors = [None] * len(htmls)  # Lista de listas, todas del tamaño del vocabulario.

for i in range(len(corpus_matrix_)):
    docs_vectors[i] = wtf(vocabulary_, corpus_matrix_[i])
    export_vector("wtf", i, titles[i], docs_vectors[i])
    
print("****** Vectores documentos (lista de listas, o matriz, con todos los vectores documento)")
print(docs_vectors)

In [ ]:
# INVERSE DOCUMENT FREQUENCY REPRESENTATIONS (bin_idf)

docs_vectors = [None] * len(htmls)  # Lista de listas, todas del tamaño del vocabulario.

for i in range(len(corpus_matrix_)):
    doc_dict = corpus_matrix_[i]
    print(i)
    docs_vectors[i] = bin_idf(docs_frequencies_, doc_dict)
    export_vector("bin_idf", i, titles[i], docs_vectors[i])
    
print("****** Vectores documentos (lista de listas, o matriz, con todos los vectores documento)")
print(docs_vectors)
    


In [ ]:
# TERM FREQUENCY - INVERSE DOCUMENT FREQUENCY REPRESENTATIONS (tf_idf)

docs_vectors = [None] * len(htmls)  # Lista de listas, todas del tamaño del vocabulario.

for i in range(len(corpus_matrix_)):
    doc_dict = corpus_matrix_[i]
    print(i)
    docs_vectors[i] = tf_idf(docs_frequencies_, doc_dict)
    export_vector("tf_idf", i, titles[i], docs_vectors[i])

print("****** Vectores documentos (lista de listas, o matriz, con todos los vectores documento)")
print(docs_vectors)

In [ ]:

currentDirectory = os.getcwd()
currentDirectory

## Códigos originales de internet

In [ ]:
# https://stackoverflow.com/questions/1936466/beautifulsoup-grab-visible-webpage-text

from bs4 import BeautifulSoup
from bs4.element import Comment
import urllib.request


def tag_visible(element):
    if element.parent.name in ['style', 'script', 'head', 'title', 'meta', '[document]']:
        return False
    if isinstance(element, Comment):
        return False
    return True


def text_from_html(body):
    soup = BeautifulSoup(body, 'html.parser')
    texts = soup.findAll(text=True)
    visible_texts = filter(tag_visible, texts)  
    return u" ".join(t.strip() for t in visible_texts)

html = urllib.request.urlopen('http://www.nytimes.com/2009/12/21/us/21storm.html').read()
print(text_from_html(html))

In [ ]:
import requests
from bs4 import BeautifulSoup

url = 'https://www.troyhunt.com/the-773-million-record-collection-1-data-reach/'
res = requests.get(url)
html_page = res.content
soup = BeautifulSoup(html_page, 'html.parser')
text = soup.find_all(text=True)

output = ''
blacklist = [
	'[document]',
	'noscript',
	'header',
	'html',
	'meta',
	'head', 
	'input',
	'script',
	# there may be more elements you don't want, such as "style", etc.
]

for t in text:
	if t.parent.name not in blacklist:
		output += '{} '.format(t)

print(output)